# Session XIX:
## Advanced Lab — Data Explorer and Visualizer with API Integration

This is the **capstone lab** of Python I. You will build a complete data exploration
application that fetches live data from a public API, stores it locally, analyzes it
with DuckDB, and visualizes it through a Streamlit dashboard.

**Important:** This lab combines exercises into `.py` module files. Each exercise
below asks you to implement a specific module. At the end, you will wire them
together into a working Streamlit application.

---

## Exercise 1: Data Fetcher Module (`src/fetcher.py`)

**Objective:** Build a module that fetches data from the REST Countries API and
caches it locally to avoid repeated API calls.

**Requirements:**
1. Write a function `fetch_countries(url)` that:
   - Sends a GET request with a 10-second timeout
   - Raises appropriate errors for HTTP failures
   - Returns the parsed JSON response
2. Write a function `fetch_with_cache(url, cache_path, force_refresh)` that:
   - Returns cached data from `cache_path` if the file exists and `force_refresh` is `False`
   - Otherwise fetches from the API, saves to `cache_path`, and returns the data
   - Creates parent directories if they don't exist
3. Both functions must have **type hints** and **docstrings**
4. Handle `Timeout`, `HTTPError`, and `JSONDecodeError` exceptions separately

In [ ]:
# Save this as: src/fetcher.py

import requests
import json
import os
from typing import Optional

BASE_URL = "https://restcountries.com/v3.1/all"


def fetch_countries(url: str = BASE_URL) -> list[dict]:
    """Fetch country data from the REST Countries API.

    Args:
        url: The API endpoint URL.

    Returns:
        List of country dictionaries.
    """
    # TODO: Implement the function
    #   1. Send a GET request with timeout=10
    #   2. Call raise_for_status()
    #   3. Return the parsed JSON
    #   4. Handle Timeout, HTTPError, and JSONDecodeError separately
    pass


def fetch_with_cache(
    url: str = BASE_URL,
    cache_path: str = "data/raw/countries.json",
    force_refresh: bool = False
) -> list[dict]:
    """Fetch data from API or load from local cache.

    Args:
        url: The API endpoint.
        cache_path: Path to the local cache file.
        force_refresh: If True, ignore cache and re-fetch.

    Returns:
        List of country dictionaries.
    """
    # TODO: Implement the function
    #   1. If cache_path exists AND force_refresh is False, load and return from file
    #   2. Otherwise, call fetch_countries(url)
    #   3. Create parent directories with os.makedirs(..., exist_ok=True)
    #   4. Save the data to cache_path as JSON
    #   5. Return the data
    pass

---

## Exercise 2: Data Processing Module (`src/processor.py`)

**Objective:** Build a module that transforms raw API data into clean, flat
records suitable for CSV storage and DuckDB analysis.

**Requirements:**
1. Write a function `flatten_country(country)` that extracts:
   - `name` (from `country["name"]["common"]`)
   - `official_name` (from `country["name"]["official"]`)
   - `region` (default: `"Unknown"`)
   - `subregion` (default: `"Unknown"`)
   - `population` (default: `0`)
   - `area` (default: `0.0`)
   - `capital` (first capital or `"N/A"`)
   - `languages` (comma-separated string of all languages, or `"N/A"`)
   - `currencies` (comma-separated string of currency names, or `"N/A"`)
2. Write a function `process_all(raw_data)` that applies `flatten_country` to
   every item, skipping any that raise exceptions
3. Write a function `save_processed_csv(records, filepath)` using `csv.DictWriter`
4. All functions must handle missing/malformed data gracefully with `.get()`

In [ ]:
# Save this as: src/processor.py

import csv
from typing import Optional


def flatten_country(country: dict) -> dict:
    """Extract and flatten relevant fields from a raw country record.

    Args:
        country: A single country dictionary from the API.

    Returns:
        A flat dictionary with standardized fields.
    """
    # TODO: Implement the function
    #   Use .get() with defaults for every field
    #   Handle nested structures: name.common, name.official
    #   Join languages dict values into a comma-separated string
    #   Join currencies dict values (each has a "name" key) into a string
    #   Get the first capital from the capitals list, or "N/A"
    pass


def process_all(raw_data: list[dict]) -> list[dict]:
    """Process all raw country records into flat dictionaries.

    Args:
        raw_data: List of raw country dicts from the API.

    Returns:
        List of flattened country dictionaries.
    """
    # TODO: Implement the function
    #   Loop through raw_data
    #   Call flatten_country on each
    #   Use try/except to skip records that fail
    #   Print a warning for skipped records
    pass


def save_processed_csv(records: list[dict], filepath: str) -> None:
    """Save processed records to a CSV file.

    Args:
        records: List of flat country dictionaries.
        filepath: Output CSV file path.
    """
    # TODO: Implement the function
    #   Handle empty records list
    #   Use csv.DictWriter with fieldnames from the first record
    #   Write header and all rows
    pass

---

## Exercise 3: Analysis Module (`src/analyzer.py`)

**Objective:** Build a module that runs DuckDB SQL queries against the
processed CSV file to answer analytical questions.

**Requirements:**
1. Write a function `get_region_summary(csv_path)` that returns:
   - Region name, country count, total population, average area
   - Ordered by total population descending
2. Write a function `get_top_countries(csv_path, n, metric)` that returns:
   - Top N countries by the given metric (`population` or `area`)
   - Returns name, region, and the metric value
3. Write a function `search_countries(csv_path, query)` that:
   - Searches for countries whose name contains the query string (case-insensitive)
   - Returns name, region, population, capital
4. Write a function `get_language_stats(csv_path)` that:
   - Counts how many countries list each language
   - Returns the top 15 languages
5. All functions must return results as **lists of dictionaries** (not raw tuples)

In [ ]:
# Save this as: src/analyzer.py

import duckdb


def _query_as_dicts(sql: str) -> list[dict]:
    """Execute a SQL query and return results as a list of dicts."""
    con = duckdb.connect()
    result = con.execute(sql)
    columns = [desc[0] for desc in result.description]
    rows = result.fetchall()
    con.close()
    return [dict(zip(columns, row)) for row in rows]


def get_region_summary(csv_path: str = "data/processed/countries.csv") -> list[dict]:
    """Get summary statistics grouped by region.

    Returns:
        List of dicts with: region, country_count, total_population, avg_area
    """
    # TODO: Write the SQL query and call _query_as_dicts
    #   SELECT region, COUNT, SUM(population), ROUND(AVG(area), 1)
    #   GROUP BY region, ORDER BY total_population DESC
    pass


def get_top_countries(
    csv_path: str = "data/processed/countries.csv",
    n: int = 10,
    metric: str = "population"
) -> list[dict]:
    """Get the top N countries by a given metric.

    Args:
        csv_path: Path to the processed CSV.
        n: Number of results to return.
        metric: Column to sort by ("population" or "area").

    Returns:
        List of dicts with: name, region, and the metric value.
    """
    # TODO: Write a dynamic SQL query using the metric parameter
    #   Validate that metric is either "population" or "area"
    #   SELECT name, region, {metric} FROM ... ORDER BY {metric} DESC LIMIT {n}
    pass


def search_countries(
    csv_path: str = "data/processed/countries.csv",
    query: str = ""
) -> list[dict]:
    """Search countries by name (case-insensitive).

    Args:
        csv_path: Path to the processed CSV.
        query: Search string.

    Returns:
        List of dicts with: name, region, population, capital.
    """
    # TODO: Use ILIKE for case-insensitive search
    #   WHERE name ILIKE '%{query}%'
    pass


def get_language_stats(
    csv_path: str = "data/processed/countries.csv"
) -> list[dict]:
    """Count how many countries list each language.

    Returns:
        Top 15 languages with: language, country_count.
    """
    # TODO: Use UNNEST(string_split(languages, ', ')) to explode languages
    #   Then GROUP BY and COUNT, ORDER BY count DESC, LIMIT 15
    pass

---

## Exercise 4: Streamlit Dashboard (`app.py`)

**Objective:** Build the main Streamlit application that wires together
all modules into an interactive dashboard.

**Requirements:**
1. **Page config:** Set title, wide layout, and a page icon
2. **Sidebar filters:**
   - A selectbox to filter by region (including an "All" option)
   - A slider for minimum population
   - A text input for country name search
   - A button to force-refresh data from the API
3. **Metrics row:** Show 3 `st.metric` cards:
   - Total countries (after filtering)
   - Total population
   - Average area
4. **Two tabs:**
   - Tab 1: Bar chart showing country count by region
   - Tab 2: Interactive data table with full filtered data
5. **Expander section:** Show top 10 countries by population
6. Use `@st.cache_data` to cache the data loading pipeline
7. Import and use functions from `src/fetcher.py`, `src/processor.py`, and `src/analyzer.py`

In [ ]:
# Save this as: app.py

import streamlit as st

# TODO: Import your modules
# from src.fetcher import fetch_with_cache
# from src.processor import process_all, save_processed_csv
# from src.analyzer import get_region_summary, get_top_countries, search_countries

# ── Constants ──
API_URL = "https://restcountries.com/v3.1/all"
RAW_PATH = "data/raw/countries.json"
CSV_PATH = "data/processed/countries.csv"


# TODO: Step 1 — Page configuration
# st.set_page_config(...)


# TODO: Step 2 — Cached data loading function
# @st.cache_data(ttl=3600)
# def load_data(force_refresh=False):
#     raw = fetch_with_cache(API_URL, RAW_PATH, force_refresh)
#     records = process_all(raw)
#     save_processed_csv(records, CSV_PATH)
#     return records


# TODO: Step 3 — Sidebar filters
# with st.sidebar:
#     st.header("Filters")
#     ... selectbox, slider, text_input, button ...


# TODO: Step 4 — Load and filter data
# data = load_data()
# Apply region filter, population filter, search filter


# TODO: Step 5 — Metrics row
# col1, col2, col3 = st.columns(3)
# col1.metric(...)


# TODO: Step 6 — Tabs with chart and table
# tab1, tab2 = st.tabs([...])


# TODO: Step 7 — Expander with top countries
# with st.expander("..."):
#     ...

---

## Exercise 5: Data Export Feature

**Objective:** Add a download feature to your Streamlit dashboard that lets
users export the currently filtered data as a CSV file.

**Requirements:**
1. Write a function `records_to_csv_string(records)` in `src/processor.py` that:
   - Takes a list of dictionaries
   - Returns a CSV-formatted **string** (not a file) using `io.StringIO`
2. Add a `st.download_button` to the dashboard sidebar that:
   - Is labeled "📥 Download Filtered Data"
   - Downloads the currently filtered records as `filtered_countries.csv`
   - Uses MIME type `text/csv`
3. Show the number of records that will be downloaded above the button

In [ ]:
# Add this function to: src/processor.py

import io
import csv


def records_to_csv_string(records: list[dict]) -> str:
    """Convert a list of dictionaries to a CSV-formatted string.

    Args:
        records: List of flat dictionaries.

    Returns:
        A string containing the CSV content.
    """
    # TODO: Implement the function
    #   1. Handle empty records list (return empty string)
    #   2. Create an io.StringIO buffer
    #   3. Use csv.DictWriter to write header + rows to the buffer
    #   4. Return buffer.getvalue()
    pass


# Add this to app.py sidebar:
# csv_string = records_to_csv_string(filtered_data)
# st.sidebar.write(f"Records to export: {len(filtered_data)}")
# st.sidebar.download_button(
#     label="📥 Download Filtered Data",
#     data=csv_string,
#     file_name="filtered_countries.csv",
#     mime="text/csv"
# )

---

## Exercise 6: Complete Integration and Git Packaging (Capstone)

**Objective:** Wire everything together into a fully working application,
then package it as a professional GitHub repository.

**Requirements:**

### Part A — Integration
1. Ensure all modules (`fetcher.py`, `processor.py`, `analyzer.py`) work together
2. The `app.py` dashboard must:
   - Load data (from cache or API) on startup
   - Apply all sidebar filters dynamically
   - Show metrics, charts, tables, and download button
   - Handle errors gracefully (show `st.error()` instead of crashing)
3. Add an `st.expander("Language Statistics")` section using `get_language_stats()`

### Part B — Project Packaging
1. Create a proper `pyproject.toml` with all dependencies
2. Write a `README.md` with:
   - Project description
   - How to install (`uv sync`)
   - How to run (`uv run streamlit run app.py`)
   - Screenshot of the dashboard (optional)
3. Create a `.gitignore` that excludes:
   - `.venv/`, `__pycache__/`, `data/raw/` (but keep `data/processed/`)
4. Make at least **5 meaningful git commits** with descriptive messages
5. Push to a GitHub repository

In [ ]:
# This exercise is a project — no starter code.
# Use the following checklist to track your progress:

checklist = {
    "src/fetcher.py works standalone": False,
    "src/processor.py works standalone": False,
    "src/analyzer.py works standalone": False,
    "app.py runs with streamlit": False,
    "Sidebar filters work": False,
    "Metrics update dynamically": False,
    "Charts render correctly": False,
    "Download button exports CSV": False,
    "Language stats expander works": False,
    "Error handling (no crashes)": False,
    "pyproject.toml complete": False,
    "README.md written": False,
    ".gitignore configured": False,
    "5+ git commits": False,
    "Pushed to GitHub": False,
}

for task, done in checklist.items():
    status = "✅" if done else "☐"
    print(f"  {status} {task}")

---

## Summary

| Exercise | Module | Key Skills |
|----------|--------|------------|
| 1 | `src/fetcher.py` | `requests`, caching, error handling |
| 2 | `src/processor.py` | Data flattening, `.get()`, CSV writing |
| 3 | `src/analyzer.py` | DuckDB SQL, dynamic queries, aggregation |
| 4 | `app.py` | Streamlit layout, filters, charts, metrics |
| 5 | Export feature | `io.StringIO`, `st.download_button` |
| 6 | Full integration | Git, packaging, README, deployment |

**Run your final application:**
```bash
$ uv run streamlit run app.py
```